# 03 — Snore Denoising Exploration

Tune the denoiser on a real overnight recording and view raw-vs-clean spectrograms.

**Pipeline:** decode an m4a slice → find a noise-floor clip → `denoise()` with a preset → visualize / listen.

Recordings live outside the repo in the sibling `Recordings/` folder (AAC mono 48 kHz, 4–6 h); set `SNORE_RECORDINGS` to point elsewhere. We only ever load short *slices* — never a whole multi-hour file. Decoding needs the ffmpeg bundled by `imageio-ffmpeg` (libsndfile can't read AAC).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # so `import src.*` works from notebooks/

import numpy as np
import soundfile as sf
from IPython.display import Audio, display
import matplotlib.pyplot as plt

from src.denoise import load_slice, denoise, PRESETS, SAMPLE_RATE as SR
from src import visualize as viz

# Recordings live outside the repo (sibling of the SnoreLab root). Override with
# the SNORE_RECORDINGS env var if yours are elsewhere.
RECORDINGS = os.environ.get('SNORE_RECORDINGS', os.path.abspath('../../../Recordings'))
FILE = os.path.join(RECORDINGS, 'Voice 260609_230949.m4a')   # loudest sample
print('SR =', SR, '| file exists:', os.path.exists(FILE))

## 1. Find active (snore) vs quiet (noise) regions

Scan the whole file cheaply at 1 kHz and look at 30 s RMS windows. Loud plateaus = snoring (often AC-on); quiet valleys = the noise floor we feed to the denoiser as its noise profile.

In [ ]:
scan = load_slice(FILE, 0, None, sr=1000)          # full file at 1 kHz (cheap)
w = 30 * 1000
rms = np.array([np.sqrt(np.mean(scan[i*w:(i+1)*w]**2)) for i in range(len(scan)//w)])
plt.figure(figsize=(14, 3))
plt.plot(np.arange(len(rms)) * 0.5, rms * 1000)    # x in minutes
plt.xlabel('minutes'); plt.ylabel('RMS ×1000'); plt.title('Energy profile — plateaus=snore, valleys=noise floor')
plt.grid(alpha=0.3); plt.show()
loud_min = float(np.argmax(rms) * 0.5)
quiet_min = float(np.argmin(rms) * 0.5)
print(f'loudest window @ {loud_min:.1f} min | quietest @ {quiet_min:.1f} min')

## 2. Decode an active slice + a noise-floor slice

In [ ]:
ACTIVE_OFFSET = 40 * 60   # seconds — tweak using the profile above (loud_min*60)
ACTIVE_DUR    = 60
NOISE_OFFSET  = 85 * 60   # a quiet valley (quiet_min*60)
NOISE_DUR     = 30

raw   = load_slice(FILE, ACTIVE_OFFSET, ACTIVE_DUR)
noise = load_slice(FILE, NOISE_OFFSET, NOISE_DUR)
print('raw', raw.shape, '| noise', noise.shape)
print(viz.band_energy_table(raw, noise, SR))   # SNR read-out per band

## 3. Inspect the noise structure (where is the AC / fan / mains hum?)

In [ ]:
viz.psd_compare(raw, noise, SR, mains_hz=50.0); plt.show()

## 4. Denoise + compare presets

`gentle` (least artifacts — ML training) / `medium` (visualization) / `aggressive` (+gate — human listening).

In [ ]:
cleaned = {name: denoise(raw, SR, name, noise_clip=noise) for name in PRESETS}
viz.presets_grid(raw, cleaned, SR); plt.show()

## 5. Headline raw-vs-clean view + listen

In [ ]:
PRESET = 'aggressive'   # change to taste
viz.raw_vs_clean(raw, cleaned[PRESET], SR, clean_title=f'CLEANED ({PRESET})'); plt.show()

print('RAW:');             display(Audio(raw, rate=SR))
print(f'CLEANED ({PRESET}):'); display(Audio(cleaned[PRESET], rate=SR))

## 6. Custom tuning (override the preset)

Build a `DenoiseConfig` by hand to push knobs — e.g. a harder gate or stronger spectral reduction — then re-listen.

In [ ]:
from src.denoise import DenoiseConfig

cfg = DenoiseConfig(highpass_hz=70, notch=True, stationary=False,
                    prop_decrease=0.92, gate=True, gate_threshold_db=-44.0)
custom = denoise(raw, SR, cfg, noise_clip=noise)
viz.raw_vs_clean(raw, custom, SR, clean_title='CLEANED (custom)'); plt.show()
display(Audio(custom, rate=SR))

# Save the cleaned slice if you like it:
# sf.write('../output/denoise_diag/clean_custom.wav', custom, SR)